In [1]:
!nvidia-smi

Thu Oct  8 10:27:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [3]:
!git clone https://github.com/amandogra2802/Text-Summarizer-Project.git

Cloning into 'Text-Summarizer-Project'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 34 (delta 6), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 6.87 KiB | 6.87 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [4]:
%cd Text-Summarizer-Project

/content/Text-Summarizer-Project


In [5]:
!ls

app.py	Dockerfile  main.py	 README.md	   research  src
config	LICENSE     params.yaml  requirements.txt  setup.py  template.py


In [6]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.8 MB/s eta 0:00:00


In [7]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.18.0
Uninstalling transformers-5.18.0:
  Successfully uninstalled transformers-5.18.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.8 MB/s eta 0:00:00


In [9]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
import evaluate

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

ModuleNotFoundError: No module named 'evaluate'

In [10]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [11]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
import evaluate

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [13]:
model_ckpt = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.28GB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [14]:
# download & unzip data

!wget https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
!unzip summarizer-data.zip

--2026-10-08 10:57:32--  https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip [following]
--2026-10-08 10:57:33--  https://raw.githubusercontent.com/entbappy/Branching-tutorial/master/summarizer-data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7903594 (7.5M) [application/zip]
Saving to: ‘summarizer-data.zip’

summarizer-data.zip 100%[===================>]   7.54M  --.-KB/s    in 0.06s   

2026-10-08 10:57:33 (121 MB/s) - ‘summarizer-data.zip’ saved [7903594/7903

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
dataset_samsum = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [18]:
from datasets import load_dataset

dataset_billsum = load_dataset("FiscalNote/billsum")

print(dataset_billsum)

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/ca_test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})


In [19]:
print(dataset_billsum["train"].column_names)

['text', 'summary', 'title']


In [20]:
print("LEGAL DOCUMENT:")
print(dataset_billsum["train"][0]["text"])

print("\nSUMMARY:")
print(dataset_billsum["train"][0]["summary"])

LEGAL DOCUMENT:
SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT ORGANIZATIONS.

    (a) Definitions.--In this section:
            (1) Business entity.--The term ``business entity'' means a 
        firm, corporation, association, partnership, consortium, joint 
        venture, or other form of enterprise.
            (2) Facility.--The term ``facility'' means any real 
        property, including any building, improvement, or appurtenance.
            (3) Gross negligence.--The term ``gross negligence'' means 
        voluntary and conscious conduct by a person with knowledge (at 
        the time of the conduct) that the conduct is likely to be 
        harmful to the health or well-being of another person.
            (4) Intentional misconduct.--The term ``intentional 
        misconduct'' means conduct by a person with knowledge (at the 
        time of the conduct) that the conduct is harmful to the health 
        or well-being 

In [23]:
model_ckpt = "facebook/bart-large-cnn"

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [2]:
from datasets import load_dataset

dataset_billsum = load_dataset("FiscalNote/billsum")

dataset_billsum

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/ca_test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})

In [3]:
print(dataset_billsum)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title'],
        num_rows: 1237
    })
})


In [4]:
print(dataset_billsum["train"].column_names)

['text', 'summary', 'title']


In [5]:
print("LEGAL DOCUMENT:")
print(dataset_billsum["train"][0]["text"])

print("\nSUMMARY:")
print(dataset_billsum["train"][0]["summary"])

LEGAL DOCUMENT:
SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES 
              TO NONPROFIT ORGANIZATIONS.

    (a) Definitions.--In this section:
            (1) Business entity.--The term ``business entity'' means a 
        firm, corporation, association, partnership, consortium, joint 
        venture, or other form of enterprise.
            (2) Facility.--The term ``facility'' means any real 
        property, including any building, improvement, or appurtenance.
            (3) Gross negligence.--The term ``gross negligence'' means 
        voluntary and conscious conduct by a person with knowledge (at 
        the time of the conduct) that the conduct is likely to be 
        harmful to the health or well-being of another person.
            (4) Intentional misconduct.--The term ``intentional 
        misconduct'' means conduct by a person with knowledge (at the 
        time of the conduct) that the conduct is harmful to the health 
        or well-being 

In [6]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch["text"],
        max_length=1024,
        truncation=True
    )

    target_encodings = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

In [7]:
if 'tokenizer' not in globals():
    from transformers import AutoTokenizer
    model_ckpt = "facebook/bart-large-cnn"
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

tokenized_dataset = dataset_billsum.map(
    convert_examples_to_features,
    batched=True
)

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [8]:
from transformers import AutoTokenizer

model_ckpt = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [9]:
print(tokenizer)

RobertaTokenizer(name_or_path='facebook/bart-large-cnn', vocab_size=50265, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
})


In [10]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(
        example_batch["text"],
        max_length=1024,
        truncation=True
    )

    target_encodings = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

In [11]:
tokenized_dataset = dataset_billsum.map(
    convert_examples_to_features,
    batched=True
)

Map:   0%|          | 0/18949 [00:00<?, ? examples/s]

Map:   0%|          | 0/3269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1237 [00:00<?, ? examples/s]

In [12]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1237
    })
})


In [13]:
print(tokenized_dataset["train"][0].keys())

dict_keys(['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'])


In [14]:
print(len(tokenized_dataset["train"]))
print(len(tokenized_dataset["test"]))
print(len(tokenized_dataset["validation"]))

18949
3269


KeyError: 'validation'

In [15]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 18949
    })
    test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1237
    })
})


In [16]:
print(tokenized_dataset.keys())

dict_keys(['train', 'test', 'ca_test'])


In [17]:
print("Train:", len(tokenized_dataset["train"]))
print("Test:", len(tokenized_dataset["test"]))
print("California Test:", len(tokenized_dataset["ca_test"]))

Train: 18949
Test: 3269
California Test: 1237


In [19]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_bart
)

NameError: name 'model_bart' is not defined

In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_ckpt = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

NameError: name 'device' is not defined

In [21]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cuda


In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_ckpt = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [23]:
print(type(model_bart))
print(device)

<class 'transformers.models.bart.modeling_bart.BartForConditionalGeneration'>
cuda


In [24]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_bart
)

In [25]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-billsum",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

In [26]:
from datasets import DatasetDict

split_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

tokenized_dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
    "test": tokenized_dataset["test"],
    "ca_test": tokenized_dataset["ca_test"]
})

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 17054
    })
    validation: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1895
    })
    test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1237
    })
})


In [28]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_bart
)

In [29]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-billsum",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    predict_with_generate=True,
    report_to="none"
)

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [30]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-billsum",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    predict_with_generate=True,
    report_to="none"
)

In [31]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_bart,
    args=training_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

In [32]:
print(tokenized_dataset.keys())

dict_keys(['train', 'validation', 'test', 'ca_test'])


In [33]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_bart,
    args=training_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

In [34]:
print(trainer)

In [35]:
trainer.train()

Step,Training Loss,Validation Loss
500,6.826832,1.698928
1000,6.591658,1.584162
1500,5.948995,1.499701
2000,5.685411,1.428203
2500,5.595919,1.413853
3000,5.436835,1.342375
3500,5.260397,1.323378
4000,5.250460,1.294623


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
500,6.826832,1.698928
1000,6.591658,1.584162
1500,5.948995,1.499701
2000,5.685411,1.428203
2500,5.595919,1.413853
3000,5.436835,1.342375
3500,5.260397,1.323378
4000,5.250460,1.294623
4264,5.335498,1.293206


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4264, training_loss=5.949751862889159, metrics={'train_runtime': 4684.0579, 'train_samples_per_second': 3.641, 'train_steps_per_second': 0.91, 'total_flos': 3.695780188441805e+16, 'train_loss': 5.949751862889159, 'epoch': 1.0})

In [36]:
print(trainer.state)

TrainerState(epoch=1.0, global_step=4264, max_steps=4264, logging_steps=100, eval_steps=500, save_steps=500, train_batch_size=1, num_train_epochs=1, num_input_tokens_seen=0, total_flos=3.695780188441805e+16, log_history=[{'loss': 8.78609130859375, 'grad_norm': 21.129344940185547, 'learning_rate': 9.900000000000002e-06, 'epoch': 0.023454907939486336, 'step': 100}, {'loss': 7.295335083007813, 'grad_norm': 27.29916763305664, 'learning_rate': 1.9900000000000003e-05, 'epoch': 0.04690981587897267, 'step': 200}, {'loss': 7.083353271484375, 'grad_norm': 23.87660789489746, 'learning_rate': 2.9900000000000002e-05, 'epoch': 0.07036472381845901, 'step': 300}, {'loss': 6.996622314453125, 'grad_norm': 22.891468048095703, 'learning_rate': 3.99e-05, 'epoch': 0.09381963175794535, 'step': 400}, {'loss': 6.826831665039062, 'grad_norm': 23.813791275024414, 'learning_rate': 4.99e-05, 'epoch': 0.11727453969743169, 'step': 500}, {'eval_loss': 1.6989282369613647, 'eval_runtime': 95.1808, 'eval_samples_per_sec

In [1]:
!pip install -q evaluate
import evaluate
import torch
from tqdm.auto import tqdm

rouge = evaluate.load("rouge")

print("ROUGE loaded successfully")

ModuleNotFoundError: No module named 'evaluate'

In [2]:
!pip install -q

ERROR: You must give at least one requirement to install (see "pip help install")


In [3]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


In [4]:
import evaluate
import torch
from tqdm.auto import tqdm

rouge = evaluate.load("rouge")

print("ROUGE loaded successfully")

ROUGE loaded successfully


In [5]:
def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=2,
    device=None,
    column_text="text",
    column_summary="summary"
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.eval()

    predictions = []
    references = []

    for start in tqdm(range(0, len(dataset), batch_size)):
        end = min(start + batch_size, len(dataset))
        batch = dataset[start:end]

        # Tokenize legal documents
        inputs = tokenizer(
            batch[column_text],
            max_length=1024,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # Generate summaries
        with torch.no_grad():
            summaries = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=128,
                num_beams=4,
                early_stopping=True
            )

        # Convert generated tokens back to text
        decoded_summaries = tokenizer.batch_decode(
            summaries,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        predictions.extend(decoded_summaries)
        references.extend(batch[column_summary])

    # Calculate ROUGE scores
    scores = metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    return scores

NameError: name 'device' is not defined

In [6]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cpu


In [7]:
if 'tokenized_dataset' not in globals():
    from datasets import load_dataset
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    import torch

    # 1. Load dataset
    dataset_billsum = load_dataset("FiscalNote/billsum")

    # 2. Load model and tokenizer
    model_ckpt = "facebook/bart-large-cnn"
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

    # 3. Tokenize
    def convert_examples_to_features(example_batch):
        input_encodings = tokenizer(example_batch["text"], max_length=1024, truncation=True)
        target_encodings = tokenizer(text_target=example_batch["summary"], max_length=128, truncation=True)
        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }
    tokenized_dataset = dataset_billsum.map(convert_examples_to_features, batched=True)

test_sample = tokenized_dataset["test"].select(range(100))

rouge_scores = calculate_metric_on_test_ds(
    dataset=test_sample,
    metric=rouge,
    model=model_bart,
    tokenizer=tokenizer,
    batch_size=2,
    device=device,
    column_text="text",
    column_summary="summary"
)

print(rouge_scores)

NameError: name 'tokenized_dataset' is not defined

In [8]:
import torch
import os

print("GPU available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

print("Tokenized dataset exists:", "tokenized_dataset" in globals())
print("BART model exists:", "model_bart" in globals())

checkpoint_dir = "/content/bart-billsum"
print("Training output folder exists:", os.path.exists(checkpoint_dir))

if os.path.exists(checkpoint_dir):
    print("Files:", os.listdir(checkpoint_dir))

GPU available: False
Device: cpu
Tokenized dataset exists: False
BART model exists: False
Training output folder exists: False


In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU available:", torch.cuda.is_available())
print("Device:", device)

GPU available: True
Device: cuda


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "bart-billsum" in dirs:
        print("Found model folder:", os.path.join(root, "bart-billsum"))

In [4]:
import os

model_folders = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if (
        "config.json" in files
        and (
            "model.safetensors" in files
            or "pytorch_model.bin" in files
        )
    ):
        model_folders.append(root)

print("Saved model folders:")
for folder in model_folders:
    print(folder)

Saved model folders:


In [5]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os

output_dir = "/content/drive/MyDrive/Text-Summarizer/bart-billsum"

os.makedirs(output_dir, exist_ok=True)

print("Model will be saved to:", output_dir)

Model will be saved to: /content/drive/MyDrive/Text-Summarizer/bart-billsum


In [7]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [8]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    report_to="none"
)

In [9]:
if 'trainer' not in globals():
    from datasets import load_dataset, DatasetDict
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
    import torch

    print("Initializing all components for training...")

    # 1. Device and Model setup
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_ckpt = "facebook/bart-large-cnn"
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
    model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

    # 2. Dataset loading and processing
    dataset_billsum = load_dataset("FiscalNote/billsum")
    def convert_examples_to_features(example_batch):
        input_encodings = tokenizer(example_batch["text"], max_length=1024, truncation=True)
        target_encodings = tokenizer(text_target=example_batch["summary"], max_length=128, truncation=True)
        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }
    tokenized_dataset = dataset_billsum.map(convert_examples_to_features, batched=True)

    # Train-test split for validation
    split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)
    tokenized_dataset = DatasetDict({
        "train": split_dataset["train"],
        "validation": split_dataset["test"],
        "test": tokenized_dataset["test"],
        "ca_test": tokenized_dataset["ca_test"]
    })

    # 3. Data collator
    seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_bart)

    # 4. Training Arguments
    if 'training_args' not in globals():
        output_dir = "/content/drive/MyDrive/Text-Summarizer/bart-billsum"
        training_args = Seq2SeqTrainingArguments(
            output_dir=output_dir,
            num_train_epochs=1,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=500,
            weight_decay=0.01,
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            save_total_limit=2,
            fp16=torch.cuda.is_available(),
            predict_with_generate=True,
            report_to="none"
        )

    # 5. Trainer Setup
    trainer = Seq2SeqTrainer(
        model=model_bart,
        args=training_args,
        processing_class=tokenizer,
        data_collator=seq2seq_data_collator,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"]
    )

trainer.train()

NameError: name 'trainer' is not defined

In [10]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_bart,
    args=training_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

print("Trainer created successfully!")

NameError: name 'model_bart' is not defined

In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
print("Device:", device)

GPU available: True
Device: cuda


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Check that the GPU is available
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. Enable the T4 GPU in Colab first."
    )

device = torch.device("cuda")
model_ckpt = "facebook/bart-large-cnn"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Load the BART model onto the GPU
model_bart = AutoModelForSeq2SeqLM.from_pretrained(
    model_ckpt
).to(device)

print("Device:", device)
print("Tokenizer loaded:", tokenizer is not None)
print("BART model loaded:", model_bart is not None)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Device: cuda
Tokenizer loaded: True
BART model loaded: True


In [13]:
from datasets import load_dataset, DatasetDict

# Load the legal summarization dataset
dataset_billsum = load_dataset("billsum")

# Split the training data: 90% for training, 10% for validation
train_validation = dataset_billsum["train"].train_test_split(
    test_size=0.1,
    seed=42
)

# Keep the original test sets separate
dataset_billsum = DatasetDict({
    "train": train_validation["train"],
    "validation": train_validation["test"],
    "test": dataset_billsum["test"],
    "ca_test": dataset_billsum["ca_test"]
})

# Check the splits
for split in dataset_billsum:
    print(f"{split}: {len(dataset_billsum[split])} examples")

print("Columns:", dataset_billsum["train"].column_names)

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/billsum@3d8510441c06a3d9dfb32eb0d7f80151730bcc4f/.huggingface.yaml'. Repository id must be 'namespace/name', got 'billsum'.

In [14]:
from datasets import load_dataset, DatasetDict

# Load BillSum using its full Hugging Face repository ID
raw_billsum = load_dataset("FiscalNote/billsum")

# Create a validation split from the training data
train_validation = raw_billsum["train"].train_test_split(
    test_size=0.1,
    seed=42
)

# Keep the original test sets separate
dataset_billsum = DatasetDict({
    "train": train_validation["train"],
    "validation": train_validation["test"],
    "test": raw_billsum["test"],
    "ca_test": raw_billsum["ca_test"]
})

# Verify the dataset
for split in dataset_billsum:
    print(f"{split}: {len(dataset_billsum[split])} examples")

print("Columns:", dataset_billsum["train"].column_names)

README.md:   0%|          | 0.00/7.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/ca_test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/18949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3269 [00:00<?, ? examples/s]

Generating ca_test split:   0%|          | 0/1237 [00:00<?, ? examples/s]

train: 17054 examples
validation: 1895 examples
test: 3269 examples
ca_test: 1237 examples
Columns: ['text', 'summary', 'title']


In [15]:
def convert_examples_to_features(example_batch):
    # Tokenize the legal documents
    input_encodings = tokenizer(
        example_batch["text"],
        max_length=1024,
        truncation=True
    )

    # Tokenize the reference summaries
    target_encodings = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True
    )

    return {
        "input_ids": input_encodings["input_ids"],
        "attention_mask": input_encodings["attention_mask"],
        "labels": target_encodings["input_ids"]
    }

In [16]:
tokenized_dataset = dataset_billsum.map(
    convert_examples_to_features,
    batched=True,
    remove_columns=dataset_billsum["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/17054 [00:00<?, ? examples/s]

Map:   0%|          | 0/1895 [00:00<?, ? examples/s]

Map:   0%|          | 0/3269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1237 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 17054
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1895
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1237
    })
})


In [17]:
tokenized_dataset = dataset_billsum.map(
    convert_examples_to_features,
    batched=True,
    remove_columns=dataset_billsum["train"].column_names
)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 17054
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1895
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3269
    })
    ca_test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1237
    })
})


In [18]:
print("Available splits:", list(tokenized_dataset.keys()))

for split in tokenized_dataset:
    print(f"{split}: {len(tokenized_dataset[split])} examples")

print("Features:", tokenized_dataset["train"].column_names)

Available splits: ['train', 'validation', 'test', 'ca_test']
train: 17054 examples
validation: 1895 examples
test: 3269 examples
ca_test: 1237 examples
Features: ['input_ids', 'attention_mask', 'labels']


In [19]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_bart
)

print("Data collator created successfully!")

Data collator created successfully!


In [20]:
import os
import torch
from transformers import Seq2SeqTrainingArguments

output_dir = "/content/drive/MyDrive/Text-Summarizer/bart-billsum"

os.makedirs(output_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    report_to="none"
)

print("Training arguments configured successfully!")

Training arguments configured successfully!


In [21]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_bart,
    args=training_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)

print("Trainer created successfully!")

Trainer created successfully!


In [22]:
/content/drive/MyDrive/Text-Summarizer/bart-billsum

NameError: name 'content' is not defined

In [23]:
train_result = trainer.train()

Step,Training Loss,Validation Loss
500,6.812061,1.696318
1000,6.591422,1.576386
1500,5.969510,1.498517
2000,5.693681,1.439246
2500,5.606237,1.406279
3000,5.440197,1.345918
3500,5.265961,1.325948
4000,5.245048,1.302328
4264,5.339127,1.295603


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [24]:
import os

final_model_dir = (
    "/content/drive/MyDrive/Text-Summarizer/bart-billsum/final"
)

os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("Final BART model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final BART model and tokenizer saved successfully!


In [25]:
import os

final_model_dir = (
    "/content/drive/MyDrive/Text-Summarizer/bart-billsum/final"
)

os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!


In [26]:
import os

print(os.listdir(final_model_dir))

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


In [27]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [28]:
import evaluate

rouge = evaluate.load("rouge")

print("ROUGE loaded successfully!")

ROUGE loaded successfully!


In [29]:
import torch
from tqdm.auto import tqdm

def calculate_rouge(
    dataset,
    model,
    tokenizer,
    metric,
    batch_size=1
):
    model.eval()

    predictions = []
    references = []
    device = next(model.parameters()).device

    for start in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[start:start + batch_size]

        inputs = tokenizer(
            batch["text"],
            max_length=1024,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            generated_tokens = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=2,
                do_sample=False
            )

        decoded_predictions = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True
        )

        predictions.extend(decoded_predictions)
        references.extend(batch["summary"])

    scores = metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    return scores

In [30]:
test_sample = dataset_billsum["test"].select(range(100))

rouge_scores = calculate_rouge(
    dataset=test_sample,
    model=model_bart,
    tokenizer=tokenizer,
    metric=rouge,
    batch_size=1
)

print("\nROUGE Scores:")
for metric_name, score in rouge_scores.items():
    print(f"{metric_name}: {score:.4f}")

  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


ROUGE Scores:
rouge1: 0.4935
rouge2: 0.3109
rougeL: 0.3813
rougeLsum: 0.4209


In [31]:
final_rouge_scores = calculate_rouge(
    dataset=dataset_billsum["test"],
    model=model_bart,
    tokenizer=tokenizer,
    metric=rouge,
    batch_size=1
)

print("Final ROUGE scores on the complete test set:")

for metric_name, score in final_rouge_scores.items():
    print(f"{metric_name}: {score:.4f}")

  0%|          | 0/3269 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Final ROUGE scores on the complete test set:
rouge1: 0.4837
rouge2: 0.2996
rougeL: 0.3674
rougeLsum: 0.4042


In [32]:
import json
import os

results_path = (
    "/content/drive/MyDrive/Text-Summarizer/"
    "bart-billsum/rouge_test_scores.json"
)

os.makedirs(os.path.dirname(results_path), exist_ok=True)

with open(results_path, "w") as f:
    json.dump(final_rouge_scores, f, indent=4)

print("ROUGE results saved to Google Drive:", results_path)

ROUGE results saved to Google Drive: /content/drive/MyDrive/Text-Summarizer/bart-billsum/rouge_test_scores.json


In [33]:
import os
import json

# Save evaluation scores in temporary Colab storage
scores = {
    name: float(value)
    for name, value in final_rouge_scores.items()
}

with open("/content/rouge_test_scores.json", "w") as f:
    json.dump(scores, f, indent=4)

# Save the currently loaded, fine-tuned BART model
local_model_dir = "/content/bart-billsum-final"

model_bart.save_pretrained(
    local_model_dir,
    safe_serialization=True
)

tokenizer.save_pretrained(local_model_dir)

print("ROUGE scores saved locally.")
print("Model and tokenizer saved locally.")
print("Model files:", os.listdir(local_model_dir))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ROUGE scores saved locally.
Model and tokenizer saved locally.
Model files: ['tokenizer_config.json', 'model.safetensors', 'generation_config.json', 'tokenizer.json', 'config.json']


In [34]:
import torch

# Prediction settings
gen_kwargs = {
    "num_beams": 4,
    "length_penalty": 0.8,
    "max_new_tokens": 128,
    "early_stopping": True
}

# Select one legal document from the test set
sample = dataset_billsum["test"][0]

sample_text = sample["text"]
reference = sample["summary"]

# Tokenize the legal document
inputs = tokenizer(
    sample_text,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

# Move inputs to the same device as BART
inputs = {
    key: value.to(model_bart.device)
    for key, value in inputs.items()
}

# Generate a summary
model_bart.eval()

with torch.inference_mode():
    generated_tokens = model_bart.generate(
        **inputs,
        **gen_kwargs
    )

prediction = tokenizer.decode(
    generated_tokens[0],
    skip_special_tokens=True
)

# Display the results
print("LEGAL DOCUMENT:")
print(sample_text[:1500])

print("\nREFERENCE SUMMARY:")
print(reference)

print("\nBART GENERATED SUMMARY:")
print(prediction)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LEGAL DOCUMENT:
SECTION 1. ENVIRONMENTAL INFRASTRUCTURE.

    (a) Jackson County, Mississippi.--Section 219 of the Water 
Resources Development Act of 1992 (106 Stat. 4835; 110 Stat. 3757) is 
amended--
        (1) in subsection (c), by striking paragraph (5) and inserting 
    the following:
        ``(5) Jackson county, mississippi.--Provision of an alternative 
    water supply and a project for the elimination or control of 
    combined sewer overflows for Jackson County, Mississippi.''; and
        (2) in subsection (e)(1), by striking ``$10,000,000'' and 
    inserting ``$20,000,000''.
    (b) Manchester, New Hampshire.--Section 219(e)(3) of the Water 
Resources Development Act of 1992 (106 Stat. 4835; 110 Stat. 3757) is 
amended by striking ``$10,000,000'' and inserting ``$20,000,000''.
    (c) Atlanta, Georgia.--Section 219(f)(1) of the Water Resources 
Development Act of 1992 (106 Stat. 4835; 113 Stat. 335) is amended by 
striking ``$25,000,000 for''.
    (d) Paterson, Passai

In [35]:
import torch

def summarize_legal_document(
    legal_text,
    max_input_length=1024,
    max_new_tokens=128
):
    if not isinstance(legal_text, str) or not legal_text.strip():
        raise ValueError("Please provide a non-empty legal document.")

    model_bart.eval()
    device = next(model_bart.parameters()).device

    inputs = tokenizer(
        legal_text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        summary_ids = model_bart.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            length_penalty=0.8,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


print("Legal summarization function created successfully!")

Legal summarization function created successfully!


In [36]:
sample_legal_text = dataset_billsum["test"][1]["text"]

generated_summary = summarize_legal_document(
    sample_legal_text
)

print("GENERATED LEGAL SUMMARY:")
print(generated_summary)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED LEGAL SUMMARY:
Federal Forage Fee Act of 1993 - Requires all grazing operations conducted on Federal lands to be subject to all applicable Federal, State, and local laws, including but not limited to:  (1) the Animal Damage Control Act; (2) the Bankhead-Jones Farm Tenant Act; and (3) the Clean Air Act, the Endangered Species Act of 1973, the Federal Advisory Committee Act, Federal Grant and Cooperative Agreement Act of 1977, Federal Insecticide, Fungicide, and Rodenticide Act, and Federal Land Policy and Management Act of 1976. 
Directs the Secretary of the Interior to
